#Imports des librairies et chargement du dataset

In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [11]:
df_CS_final = pd.read_csv("/content/drive/MyDrive/PARIS 0224 - Data Analyst/Projets/projet3/Vital Stats - Hadi, Sylvain, Henri, Khadija, Erwan/Cancer du Sein/Dossier CS final/df_CS_final.csv")

In [12]:
display(df_CS_final.head())
df_CS_final.shape

,diagnosis,radius_mean,texture_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,radius_se,perimeter_se,area_se,radius_worst,texture_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst
0,1,17.99,10.38,0.11840,0.27760,0.3001,0.14710,1.0950,8.589,153.40,25.38,17.33,0.1622,0.6656,0.7119,0.2654
1,1,20.57,17.77,0.08474,0.07864,0.0869,0.07017,0.5435,3.398,74.08,24.99,23.41,0.1238,0.1866,0.2416,0.1860
2,1,19.69,21.25,0.10960,0.15990,0.1974,0.12790,0.7456,4.585,94.03,23.57,25.53,0.1444,0.4245,0.4504,0.2430
3,1,11.42,20.38,0.14250,0.28390,0.2414,0.10520,0.4956,3.445,27.23,14.91,26.50,0.2098,0.8663,0.6869,0.2575
4,1,20.29,14.34,0.10030,0.13280,0.1980,0.10430,0.7572,5.438,94.44,22.54,16.67,0.1374,0.2050,0.4000,0.1625


(556, 16)

In [13]:
df_CS_final["diagnosis"].value_counts(normalize=True).round(2)

diagnosis
0    0.62
1    0.38
Name: proportion, dtype: float64

#Résumé

La problématique et le jeu de données nous incite à utiliser un ou plusieurs modèles de classification

Nous devons prendre en compte plusieurs facteurs:
+ sensibilité au variations d'échelle
+ sensibilité aux outliers

On peut identifier quelques modèles pertinents pour nos prédictions:
+ Régression Logistique et SVM : Performants avec scaling, sensibles aux outliers.
+ k-NN : Performant avec scaling, sensible aux outliers.
+ Arbres de Décision et Random Forests : Moins sensibles aux outliers et au scaling, généralement robustes.
+ Gradient Boosting : Moins sensible aux outliers et au scaling, souvent très performant.

Tous ces modèles ont été testés dans les différentes études et analyses que nous avons consulté (cf Ressources)

#Rappel

+ Radius Mean: Moyenne des distances du centre aux points sur le périmètre.
+ Area Mean: Moyenne de la surface de la tumeur.
+ Compactness Mean: Moyenne de (périmètre^2 / surface - 1,0).
+ Concavity Mean: Moyenne de la gravité des parties concaves du contour.
+ Concave Points Mean: Moyenne du nombre de parties concaves du contour.
+ Area Worst: Aire la plus mauvaise (moyenne des trois plus grandes valeurs) de la tumeur.
+ Compactness Worst: Compacité la plus mauvaise (moyenne des trois plus grandes valeurs) de la tumeur.
+ Concavity Worst: Concavité la plus mauvaise (moyenne des trois plus grandes valeurs) de la tumeur.
+ Area SE: Erreur standard de la surface de la tumeur.
+ Fractal Dimension SE: Erreur standard de "l'approximation du littoral" - 1.
Symmetry Worst: Symétrie la pire (moyenne des trois plus grandes valeurs) de la tumeur.
+ Fractal Dimension Worst: Dimension fractale la pire (moyenne des trois plus grandes valeurs) de la tumeur.


#Stades et grades

Stades du cancer du sein :
+ Stade 0 : Cancer in situ, non invasif. Les cellules cancéreuses sont confinées dans les canaux ou les lobules mammaires.
+ Stade I : Tumeur de petite taille (2 cm ou moins) sans propagation aux ganglions lymphatiques.
+ Stade II : Deux types de tumeurs :
Tumeur entre 2 et 5 cm ou propagation limitée aux ganglions lymphatiques
+ Stade III : Cancer localement avancé. La tumeur peut mesurer plus de 5 cm et s'être propagée à plusieurs ganglions lymphatiques ou aux tissus environnants.
+ Stade IV : Cancer métastatique. Le cancer s'est propagé à d'autres parties du corps comme les os, les poumons, le foie ou le cerveau.

Grades du cancer du sein :
Le grade histopronostique d'Elston-Ellis est utilisé pour évaluer l'agressivité de la tumeur. Il est basé sur trois critères, chacun noté de 1 à 3 :
L'architecture cellulaire (différenciation des cellules)
La forme et la taille du noyau des cellules
L'activité mitotique (vitesse de division cellulaire)

La somme de ces notes détermine le grade :
+ Grade I (scores 3-5) : Tumeurs les moins agressives, cellules bien différenciées.
+ Grade II (scores 6-7) : Agressivité intermédiaire.
+ Grade III (scores 8-9) : Tumeurs les plus agressives, cellules peu différenciées

#Choix des variables

In [14]:
df_CS_final.columns

Index(['diagnosis', 'radius_mean', 'texture_mean', 'smoothness_mean',
       'compactness_mean', 'concavity_mean', 'concave points_mean',
       'radius_se', 'perimeter_se', 'area_se', 'radius_worst', 'texture_worst',
       'smoothness_worst', 'compactness_worst', 'concavity_worst',
       'concave points_worst'],
      dtype='object')

D'après nos visualisations nous pouvons exclure de l'analyse les variables texture_mean, texture_se, texture_worst, smoothness_mean, smoothness_se, smoothness_worst, symmetry_mean, symmetry_se, symmetry_worst, fractal_dimension_mean, fractal_dimension_se et fractal_dimension_worst soit 12 marqueurs

L'analyse statistique confirme ces observations pour les marqueurs fractal_dimension_mean, texture_se, smoothness_se, symmetry_se

Les moyennes des variables fractal_dimension_mean, texture_se, smoothness_se, symmetry_se, fractal_dimension_se ne montrent pas de différence significative entre les cas bénins et malins

Nous pouvons exclure définitivement les variables texture_se, smoothness_se, symmetry_se, fractal_dimension_mean et fractal_dimension_se

Les variables radius, perimeter et area sont interdépendantes, nous allons donc en garder une des trois et nous choisissons radius en conservant ses 3 acceptions "mean", "se", et "worst"

Pour optimiser les performances des modèles certaines colonnes qui n'étaient pas impactantes selon nos conclusions ont été conservées

#Tests modèles

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import balanced_accuracy_score, classification_report, roc_auc_score

# Remplacer les valeurs de la colonne 'diagnosis' par des valeurs numériques
df_CS_final["diagnosis"] = df_CS_final["diagnosis"].replace({"M": 1, "B": 0})

# Chargement des données
X = df_CS_final.drop('diagnosis', axis=1)
y = df_CS_final['diagnosis']

# Division des données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Scaling des caractéristiques
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modèles
models = {
    "Logistic Regression": LogisticRegression(class_weight= {0: 0.38, 1 : 0.62}),
    "k-NN": KNeighborsClassifier(weights= "distance"),
    "Random Forest": RandomForestClassifier(class_weight= {0 : 0.38, 1 : 0.62}),
    "SVM": SVC(probability=True),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

# Entraînement et évaluation
results = {}
roc_auc_scores = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, "predict_proba") else None

    balanced_accuracy_train = balanced_accuracy_score(y_train, model.predict(X_train_scaled))
    balanced_accuracy_test = balanced_accuracy_score(y_test, y_pred)
    results[name + "_train"] = balanced_accuracy_train
    results[name + "_test"] = balanced_accuracy_test

    if y_pred_proba is not None:
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        roc_auc_scores[name] = roc_auc

    print(f"\nClassification Report for {name}:\n")
    print(classification_report(y_test, y_pred))

# Affichage des résultats
print("\nBalanced Accuracy Scores:")
for name, balanced_accuracy in results.items():
    print(f"{name}: {balanced_accuracy}")

print("\nROC AUC Scores:")
for name, roc_auc in roc_auc_scores.items():
    print(f"{name}: {roc_auc}")



Classification Report for Logistic Regression:

              precision    recall  f1-score   support

           0       0.97      0.96      0.97        73
           1       0.96      0.97      0.96        66

    accuracy                           0.96       139
   macro avg       0.96      0.96      0.96       139
weighted avg       0.96      0.96      0.96       139


Classification Report for k-NN:

              precision    recall  f1-score   support

           0       0.97      0.97      0.97        73
           1       0.97      0.97      0.97        66

    accuracy                           0.97       139
   macro avg       0.97      0.97      0.97       139
weighted avg       0.97      0.97      0.97       139


Classification Report for Random Forest:

              precision    recall  f1-score   support

           0       0.97      0.96      0.97        73
           1       0.96      0.97      0.96        66

    accuracy                           0.96       139
  

Après avoir testé plusieurs modèles, nous avons établi plusieurs choses:
+ nécessité de standardiser les données. Notre choix s'est porté sur RobustScaler car moins influencé par les outliers
+ nécessité de conserver la répartition des données à travers le train test split en spécifiant le paramètre stratify qui garantit de conserver le même équilibre des données dans les jeux de train et de test
+ nécessité de rééquilibrer les jeux de données (62% bénins, 38% malins). On choisit d'utiliser la librairie imblearn et l'outil SMOTE
+ nous avons aussi paramétré l'hyper-paramètre class_weight pour les modèles Régression Logistique et RandomForestClassifier

Les modèles testés sont tous très performants mais pour réduire le risque d'overfitting nous ne conserverons pas le KNNClassifier et le XGBClassifier.
Nous nous focaliserons sur les modèles suivants:
+ Régression Logistique
+ Random Forest Classifier
+ Support Vector Classifier

Pour décider entre chaque modèle, nous afficherons le classification report, le balanced accuracy score et la matrice de confusion. Notre objectif, en testant diverses combinaisons d'hyper-paramètres est d'obtenir un score de faux positifs de 0, c'est à dire aucune tumeur maligne prédite bénigne

##Regression Logistique

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

# Remplacer les valeurs de la colonne 'diagnosis' par des valeurs numériques
df_CS_final["diagnosis"] = df_CS_final["diagnosis"].replace({"M": 1, "B": 0})

# Chargement des données
X = df_CS_final.drop('diagnosis', axis=1)
y = df_CS_final['diagnosis']

# Division des données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify = y)

# Rééquilibrage des classes avec SMOTE
smote = SMOTE()
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Scaling des caractéristiques
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

# Modèle
model_log = LogisticRegression(max_iter = 1000)

# Entraînement et évaluation
model_log.fit(X_train_scaled, y_train_resampled)

y_pred = model_log.predict(X_test_scaled)
y_pred_proba = model_log.predict_proba(X_test_scaled)[:, 1]


balanced_accuracy_train = balanced_accuracy_score(y_train_resampled, model_log.predict(X_train_scaled))
balanced_accuracy_test = balanced_accuracy_score(y_test, y_pred)


print(f"\nClassification Report pour la Regression Logistiques:\n")
print(classification_report(y_test, y_pred))


Classification Report pour la Regression Logistiques:

              precision    recall  f1-score   support

           0       0.98      0.98      0.98        86
           1       0.96      0.96      0.96        53

    accuracy                           0.97       139
   macro avg       0.97      0.97      0.97       139
weighted avg       0.97      0.97      0.97       139



In [ ]:
print("\nBalanced Accuracy Score:")
print(f"Regression Logistique sur le Train: {balanced_accuracy_train}")
print(f"Regression Logistique sur le Test: {balanced_accuracy_test}")


Balanced Accuracy Score:
Regression Logistique sur le Train: 0.9476744186046512
Regression Logistique sur le Test: 0.9593023255813953


In [ ]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(cm, index = ["Réel Bénin", "Réel Malin"], columns = ["Prédit Bénin", "Prédit Malin"])
print(cm_df)

            Prédit Bénin  Prédit Malin
Réel Bénin            79             7
Réel Malin             0            53


##Random Forest Classifier

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

In [18]:
from sklearn.model_selection import train_test_split

X = df_CS_final.drop("diagnosis", axis = 1)
y = df_CS_final["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size = 0.25)
print("The length of the initial dataset is :", len(X))
print("The length of the train dataset is   :", len(X_train))
print("The length of the test dataset is    :", len(X_test))

The length of the initial dataset is : 556
The length of the train dataset is   : 417
The length of the test dataset is    : 139


In [19]:
# Rééquilibrage des classes avec SMOTE
smote = SMOTE()
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [20]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_train = scaler.fit_transform(X_train_resampled)
X_test = scaler.transform(X_test)

In [21]:
modelRFC = RandomForestClassifier(class_weight = {0: 0.38, 1: 0.62})

modelRFC.fit(X_train, y_train_resampled)

RandomForestClassifier(class_weight={0: 0.38, 1: 0.62})

In [22]:
y_pred = modelRFC.predict(X_test)

accuracy_train = balanced_accuracy_score(y_train_resampled, modelRFC.predict(X_train))
accuracy_test = balanced_accuracy_score(y_test, y_pred)
print(f"Score de précision train: {accuracy_train}")
print(f"Score de précision test: {accuracy_test}")

Score de précision train: 1.0
Score de précision test: 0.9574512245745123


In [23]:
class_report = classification_report(y_test, y_pred)
print("Classification Report:", class_report)

Classification Report:               precision    recall  f1-score   support

           0       0.97      0.95      0.96        73
           1       0.94      0.97      0.96        66

    accuracy                           0.96       139
   macro avg       0.96      0.96      0.96       139
weighted avg       0.96      0.96      0.96       139



In [24]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(cm, index=['Réel Bénin', 'Réel Malin'], columns=['Prédit Bénin', 'Prédit Malin'])

print(cm_df)

            Prédit Bénin  Prédit Malin
Réel Bénin            69             4
Réel Malin             2            64


##SVC

In [25]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, balanced_accuracy_score, confusion_matrix

In [26]:
from sklearn.model_selection import train_test_split

X = df_CS_final.drop("diagnosis", axis = 1)
y = df_CS_final["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size = 0.25)
print("The length of the initial dataset is :", len(X))
print("The length of the train dataset is   :", len(X_train))
print("The length of the test dataset is    :", len(X_test))

The length of the initial dataset is : 556
The length of the train dataset is   : 417
The length of the test dataset is    : 139


In [27]:
# Rééquilibrage des classes avec SMOTE
smote = SMOTE()
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [28]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_train = scaler.fit_transform(X_train_resampled)
X_test = scaler.transform(X_test)

In [29]:
modelSVC_3 = SVC(kernel='rbf')  # Utilisation d'un kernel Radial Basis Function
modelSVC_3.fit(X_train, y_train_resampled)

SVC()

In [30]:
y_pred = modelSVC_3.predict(X_test)

# Évaluation
balanced_accuracy_train = balanced_accuracy_score(y_train_resampled, modelSVC_3.predict(X_train))
balanced_accuracy_test = balanced_accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

In [31]:
print("Accuracy train:", balanced_accuracy_train)
print("Accuracy test:", balanced_accuracy_test)

Accuracy train: 0.9833948339483395
Accuracy test: 0.9711498547114985


In [32]:
print("Classification Report:\n", class_report)

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.97      0.97        73
           1       0.97      0.97      0.97        66

    accuracy                           0.97       139
   macro avg       0.97      0.97      0.97       139
weighted avg       0.97      0.97      0.97       139



In [33]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(cm, index=['Réel Bénin', 'Réel Malin'], columns=['Prédit Bénin', 'Prédit Malin'])

print(cm_df)

            Prédit Bénin  Prédit Malin
Réel Bénin            71             2
Réel Malin             2            64


#Remarques

Les 3 modèles ont d'excellents scores pour les données de test et les matrices de confusion indiquent qu'il y a très peu de mauvaises prédictions

Cependant, les tests avec différents hyper paramètres permet de remarquer que la régression logistique est le modèle qui permet une meilleure précision dans les prédictions avec un léger avantage sur le RandomForestClassifier.
+ Regression Logistique: 7 erreurs sur 139 prédictions
+ RandomForestClassifier: 6 erreurs sur 139 prédictions
+ SVC: 4 erreurs sur 139 prédictions

Le modèle Régression Logistique ne fait pas de mauvaise prédiction pour les cas malins mais 7 mauvaises prédictions pour les cas bénins
Le modèle RandomForestClassifier contient 2 mauvaises prédictions pour des cas malins et 4 erreurs pour les cas bénins
Le modèle SVC fait 2 mauvaises prédictions pour les cas malins et 2 erreurs pour les cas bénins

Nous utiliserons la Régression Logistique pour la suite de notre projet car le modèle ne prédit pas bénin des cas malins

#Nouveau test Régression Logistique pour vérifier l'impact du rééquilibrage des classes

On teste différentes combinaisons pour le rééquilibrage (ou non) des classes

In [34]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

# Remplacer les valeurs de la colonne 'diagnosis' par des valeurs numériques
df_CS_final["diagnosis"] = df_CS_final["diagnosis"].replace({"M": 1, "B": 0})

# Chargement des données
X = df_CS_final.drop('diagnosis', axis=1)
y = df_CS_final['diagnosis']

# Diviser le dataset en ensembles d'entraînement et de test avec stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# Appliquer le RobustScaler
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. Sans aucune technique spéciale
model_base = LogisticRegression(random_state=42)
model_base.fit(X_train_scaled, y_train)

y_pred_base = model_base.predict(X_test_scaled)
print("Sans aucune technique spéciale - Rapport de classification:\n", classification_report(y_test, y_pred_base))
balanced_accuracy = balanced_accuracy_score(y_test, y_pred_base)
print("Balanced Accuracy Score:", balanced_accuracy)

cm = confusion_matrix(y_test, y_pred_base)
print("Confusion Matrix:\n", cm)

# 2. Avec class_weight='balanced' uniquement
model_class_weight = LogisticRegression(class_weight= "balanced", random_state=42)
model_class_weight.fit(X_train_scaled, y_train)

y_pred_class_weight = model_class_weight.predict(X_test_scaled)
print("Class_weight seul - Rapport de classification:\n", classification_report(y_test, y_pred_class_weight))
balanced_accuracy = balanced_accuracy_score(y_test, y_pred_class_weight)
print("Balanced Accuracy Score:", balanced_accuracy)

cm = confusion_matrix(y_test, y_pred_class_weight)
print("Confusion Matrix:\n", cm)

# 3. Avec SMOTE uniquement
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

model_smote = LogisticRegression(random_state=42)
model_smote.fit(X_train_resampled, y_train_resampled)

y_pred_smote = model_smote.predict(X_test_scaled)
print("SMOTE seul - Rapport de classification:\n", classification_report(y_test, y_pred_smote))
balanced_accuracy = balanced_accuracy_score(y_test, y_pred_smote)
print("Balanced Accuracy Score:", balanced_accuracy)

cm = confusion_matrix(y_test, y_pred_smote)
print("Confusion Matrix:\n", cm)

# 4. Avec SMOTE et class_weight='balanced'
model_combined = LogisticRegression(class_weight= "balanced", random_state=42)
model_combined.fit(X_train_resampled, y_train_resampled)

y_pred_combined = model_combined.predict(X_test_scaled)
print("SMOTE et class_weight - Rapport de classification:\n", classification_report(y_test, y_pred_combined))
balanced_accuracy = balanced_accuracy_score(y_test, y_pred_combined)
print("Balanced Accuracy Score:", balanced_accuracy)

cm = confusion_matrix(y_test, y_pred_combined)
print("Confusion Matrix:\n", cm)


Sans aucune technique spéciale - Rapport de classification:
               precision    recall  f1-score   support

           0       0.98      0.99      0.98        86
           1       0.98      0.96      0.97        53

    accuracy                           0.98       139
   macro avg       0.98      0.98      0.98       139
weighted avg       0.98      0.98      0.98       139

Balanced Accuracy Score: 0.975318121983326
Confusion Matrix:
 [[85  1]
 [ 2 51]]
Class_weight seul - Rapport de classification:
               precision    recall  f1-score   support

           0       0.98      0.95      0.96        86
           1       0.93      0.96      0.94        53

    accuracy                           0.96       139
   macro avg       0.95      0.96      0.95       139
weighted avg       0.96      0.96      0.96       139

Balanced Accuracy Score: 0.9578762615182097
Confusion Matrix:
 [[82  4]
 [ 2 51]]
SMOTE seul - Rapport de classification:
               precision    recall

##Remarques

On constate qu'on obtient les mêmes scores sans rééquilibrage du dataset ni hyperparamétrage de class_weight et en utilisant uniquement un rééquilibrage avec SMOTE

#Conclusion

Le modèle Régression Logistique est très performant quand il s'agit de prédire la bénignité ou la malignité d'une tumeur dans le contexte du Cancer du Sein

#Ressources

+ A Comparative Analysis of Breast Cancer Detection and Diagnosis Using Data Visualization and Machine Learning Applications, by Muhammet Fatih Ak, 26/04/2020
+ Application of Machine Learning Algorithms in Breast Cancer Diagnosis and Classification, by Clement G. Yedjou, Solange S. Tchounwou, Richard A. Aló, Rashid Elhag, BereKet Mochona, and Lekan Latinwo, 30/10/2021
+ Prediction of Breast Cancer using Machine Learning Approaches, by Reza Rabiei, Seyed Mohammad Ayyoubzadeh, Solmaz Sohrabei, Marzieh Esmaeili, and Alireza Atashi, 01/06/2022
+ Machine Learning Algorithms For Breast Cancer Prediction And
Diagnosis by Mohammed Amine Naji, Sanaa El Filalib, Kawtar Aarikac, EL Habib Benlahmard, Rachida Ait Abdelouhahide, Olivier Debauchef, 12/08/2021
+ Breast Cancer Detection and Prevention Using Machine Learning
by Arslan Khalid, Arif Mehmood, Amerah Alabrah, Bader Fahad Alkhamees, Farhan Amin, Hussain AlSalman and Gyu Sang Choi, 02/10/2023
